# PTB-XL revision notebook
Run top to bottom. Each stage saves to Drive and skips itself if its outputs exist, so after a disconnect just re-run everything.

1. Setup, 2. Splits, 3. Model definitions, 4. Inference from saved weights (M1-M5), 5. M6 fixed (train-only MiniRocket fit, SMOTE on train only, early stopping on val) plus SMOTE ablation, 6. Statistics (patient-cluster paired bootstrap, McNemar, AUROC/AUPRC, sens/spec, calibration), then figures and LaTeX tables.

## 1. Setup

In [ ]:
!pip install -q sktime lightgbm imbalanced-learn
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, time, gc, logging, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, average_precision_score
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)
log = logging.getLogger()

RES = Path("/content/drive/MyDrive/results_v3")   # same folder as before
OUT = RES / "revision"
OUT.mkdir(exist_ok=True)

SEED = 42
K = 5
CLASSES = ["CD", "HYP", "MI", "NORM", "STTC"]
RUN_NO_SMOTE_ABLATION = True   # extra LightGBM run without SMOTE
DELTA = 0.02                   # equivalence margin on macro F1. Choose and justify BEFORE looking at results
B_BOOT = 10000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"device: {device}")
with open(RES / "class_map.json") as f:
    assert json.load(f) == {c: i for i, c in enumerate(CLASSES)}, "class map changed"

## 2. Splits and data
The split is rebuilt from `patient_ids_12lead.npy` and `record_ids_raw_12lead.npy` (same shuffle, seed 42). If `X_raw_12lead.npy` or `y_raw_12lead.npy` are missing, they are rebuilt from PTB-XL (2b labels, 2c signals). The rebuilt labels are checked against the paper's exact class counts, which also pins down the multi-label rule.

In [ ]:
import re
pat = np.load(RES / "patient_ids_12lead.npy")
rec_ids = np.load(RES / "record_ids_raw_12lead.npy")
print("patient_ids:", pat.dtype, pat.shape, "| record_ids:", rec_ids.dtype, rec_ids.shape, rec_ids[:3])
assert len(pat) == len(rec_ids) == 21375

# record ids -> PTB-XL ecg_id integers
if rec_ids.dtype.kind in "iu":
    ecg_ids = rec_ids.astype(int)
else:
    ecg_ids = np.array([int(re.findall(r"\d+", str(r))[-1]) for r in rec_ids])

def make_splits(pat, seed=42):
    u = np.unique(pat)
    np.random.seed(seed)
    np.random.shuffle(u)
    n = int(len(u) * 0.15)
    test_idx = np.where(np.isin(pat, u[:n]))[0]
    val_idx = np.where(np.isin(pat, u[n:2 * n]))[0]
    train_idx = np.where(np.isin(pat, u[2 * n:]))[0]
    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = make_splits(pat, SEED)
print(len(train_idx), len(val_idx), len(test_idx))
assert (len(train_idx), len(val_idx), len(test_idx)) == (14978, 3215, 3182), "split differs from the paper"

### 2b. Labels (rebuilt only if `y_raw_12lead.npy` is missing)
Tries the plausible multi-label rules (first code, highest likelihood, every priority order) with three likelihood thresholds. A rule is accepted only if it reproduces the paper's exact test class counts (496 / 210 / 598 / 1386 / 492), the 1,293 HYP records, and the same 21,375-record set.

In [ ]:
import ast, itertools, subprocess, shutil
TEST_SUPPORT = np.array([496, 210, 598, 1386, 492])   # CD, HYP, MI, NORM, STTC from the paper's test report
TOTAL_HYP = 1293

def find(name):
    for d in (Path("/content"), OUT, RES):
        if (d / name).exists():
            return d / name
    return None

def load_db():
    meta = Path("/content/ptbxl_meta"); meta.mkdir(exist_ok=True)
    for f in ("ptbxl_database.csv", "scp_statements.csv"):
        if not (meta / f).exists():
            subprocess.run(["wget", "-q", "-O", str(meta / f),
                            f"https://physionet.org/files/ptb-xl/1.0.3/{f}"], check=True)
    d = pd.read_csv(meta / "ptbxl_database.csv", index_col="ecg_id")
    d["scp_codes"] = d["scp_codes"].apply(ast.literal_eval)
    s = pd.read_csv(meta / "scp_statements.csv", index_col=0)
    return d, s

XP, YP = find("X_raw_12lead.npy"), find("y_raw_12lead.npy")
print("X:", XP, "| y:", YP)

if YP is None:
    db, scp = load_db()
    code2cls = scp[scp["diagnostic"] == 1]["diagnostic_class"].to_dict()
    CI = {c: i for i, c in enumerate(CLASSES)}
    assert set(code2cls.values()) <= set(CI), set(code2cls.values())
    assert np.isin(ecg_ids, db.index).all(), "record_ids are not PTB-XL ecg_ids. Check rec_ids[:5]"
    N = len(ecg_ids)
    ent_all = {i: [(CI[code2cls[c]], float(l), c) for c, l in codes.items() if c in code2cls]
               for i, codes in db["scp_codes"].items()}
    target = set(ecg_ids.tolist())

    def pick(lst, tie):
        if tie == "first":
            cand = lst
        elif tie == "last":
            cand = lst[::-1]
        elif tie == "code A-Z":
            cand = sorted(lst, key=lambda t: t[2])
        else:
            cand = sorted(lst, key=lambda t: t[2], reverse=True)
        best = cand[0]
        for t in cand[1:]:
            if t[1] > best[1]:
                best = t
        return best[0]

    cands = []   # (score, thr_name, rule_name, labels, multi_share)
    for thr_name, thr in [("all codes", -1.0), ("lik>0", 0.0), ("lik>=50", 49.9)]:
        kept = {i: [(k, l, c) for k, l, c in lst if l > thr] for i, lst in ent_all.items()}
        kept = {i: lst for i, lst in kept.items() if lst}
        if set(kept) != target:
            print(f"[{thr_name}] kept {len(kept)} records, not the paper's record set -> skipped")
            continue
        lists = [kept[i] for i in ecg_ids]
        M = np.zeros((N, K), bool)
        Lmax = np.full((N, K), -1.0)
        Lsum = np.full((N, K), -1.0)
        for r, lst in enumerate(lists):
            for k, l, c in lst:
                M[r, k] = True
                Lmax[r, k] = max(Lmax[r, k], l)
                Lsum[r, k] = max(Lsum[r, k], 0.0) + l
        multi = float((M.sum(1) > 1).mean())
        rules = {"first code": np.array([lst[0][0] for lst in lists])}
        for tie in ("first", "last", "code A-Z", "code Z-A"):
            rules[f"highest-likelihood code (ties: {tie})"] = np.array([pick(l, tie) for l in lists])
        rules["class max likelihood (ties: lowest class idx)"] = np.argmax(Lmax, 1)
        rules["class max likelihood (ties: highest class idx)"] = K - 1 - np.argmax(Lmax[:, ::-1], 1)
        rules["class summed likelihood (ties: lowest class idx)"] = np.argmax(Lsum, 1)
        rules["class summed likelihood (ties: highest class idx)"] = K - 1 - np.argmax(Lsum[:, ::-1], 1)
        for perm in itertools.permutations(range(K)):
            lab = np.full(N, -1)
            for k in reversed(perm):
                lab[M[:, k]] = k
            rules["priority " + ">".join(CLASSES[k] for k in perm)] = lab
            Lp = Lmax.copy()
            for pos, k in enumerate(perm):
                Lp[:, k] += (K - pos) * 1e-3
            rules["class max likelihood, ties by " + ">".join(CLASSES[k] for k in perm)] = np.argmax(Lp, 1)
        for rname, lab in rules.items():
            sup = np.bincount(lab[test_idx], minlength=K)
            score = int(np.abs(sup - TEST_SUPPORT).sum() + abs((lab == 1).sum() - TOTAL_HYP))
            cands.append((score, thr_name, rname, lab, multi))

    cands.sort(key=lambda t: t[0])
    print("best candidates (score 0 = exact match):")
    for s, t, r, _, _ in cands[:12]:
        print(f"  score {s:4d} | {t} | {r}")
    if not cands or cands[0][0] != 0:
          s, thr_name, rule_name, y_new, multi = cands[0]
    sup = np.bincount(y_new[test_idx], minlength=K)
    print("test counts:", sup.tolist(), "| paper:", TEST_SUPPORT.tolist())
    print("HYP total:", int((y_new == 1).sum()), "| paper:", TOTAL_HYP)
    lab_first = next(c[3] for c in cands if c[2] == "highest-likelihood code (ties: first)")
    lab_last = next(c[3] for c in cands if c[2] == "highest-likelihood code (ties: last)")
    print("records whose label depends on tie-breaking:", int((lab_first != lab_last).sum()))
    if s > 15:
        raise RuntimeError("Best rule is too far from the paper's counts. See the printout above.")
    print(f"ACCEPTED{'' if s == 0 else ' (APPROXIMATE, score ' + str(s) + ')'}: {rule_name} ({thr_name}) | share of records with >1 superclass: {multi:.1%}")
    np.save(OUT / "y_raw_12lead.npy", y_new.astype(np.int64))
    json.dump(dict(rule=rule_name, likelihood_filter=thr_name, multi_label_share=multi, count_score=s),
              open(OUT / "label_rule.json", "w"), indent=2)
    YP = OUT / "y_raw_12lead.npy"

### 2c. Signals (rebuilt only if `X_raw_12lead.npy` is missing)
Downloads PTB-XL (about 2 GB), then applies the paper's preprocessing in the same order: 50 Hz notch (Q=30), 0.5-40 Hz Butterworth order 4 (both zero-phase via sosfiltfilt), per-lead z-score with a 1e-6 std floor. Rows follow `record_ids_raw_12lead.npy`, so the split indices line up. About 15-30 min. Section 4 then checks that M1-M5 reproduce the paper's F1, which confirms the rebuilt signals match.

In [ ]:
!pip install -q wfdb --no-deps
if XP is None:
    import wfdb
    from scipy.signal import iirnotch, tf2sos, butter, sosfiltfilt
    from numpy.lib.format import open_memmap
    db = load_db()[0]
    ROOT = Path("/content/ptbxl"); ROOT.mkdir(exist_ok=True)
    if not any(ROOT.glob("*/records500")):
        zp = Path("/content/ptbxl.zip")
        subprocess.run(["wget", "-q", "-c", "-O", str(zp),
                        "https://physionet.org/static/published-projects/ptb-xl/"
                        "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip"], check=True)
        subprocess.run(["unzip", "-q", "-o", str(zp), "*/records500/*", "-d", str(ROOT)], check=True)
        zp.unlink()
    base = next(ROOT.glob("*/records500")).parent
    log.info(f"PTB-XL at {base}")

    sos_n = tf2sos(*iirnotch(50.0, 30.0, fs=500.0))
    sos_b = butter(4, [0.5, 40.0], btype="bandpass", fs=500.0, output="sos")
    def prep(sig):
        x = sosfiltfilt(sos_n, sig, axis=-1)
        x = sosfiltfilt(sos_b, x, axis=-1)
        mu = x.mean(-1, keepdims=True)
        sd = np.maximum(x.std(-1, keepdims=True), 1e-6)
        return ((x - mu) / sd).astype(np.float32)

    CH = 1000
    CHD = OUT / "X_chunks"; CHD.mkdir(exist_ok=True)
    n_ch = (len(ecg_ids) + CH - 1) // CH
    t0 = time.time()
    for c in range(n_ch):
        cp = CHD / f"chunk_{c:03d}.npy"
        if cp.exists():
            continue
        ids_c = ecg_ids[c * CH:(c + 1) * CH]
        arr = np.empty((len(ids_c), 12, 5000), np.float32)
        for j, eid in enumerate(ids_c):
            rec = wfdb.rdrecord(str(base / db.at[eid, "filename_hr"]))
            arr[j] = prep(rec.p_signal.T.astype(np.float64))
        np.save(str(cp) + ".tmp.npy", arr)
        os.replace(str(cp) + ".tmp.npy", cp)      # a chunk only counts once fully written
        log.info(f"chunk {c + 1}/{n_ch} done ({(time.time() - t0) / 60:.1f} min this session)")

    log.info("assembling chunks ...")
    Xloc = Path("/content/X_raw_12lead.npy")
    Xm = open_memmap(Xloc, mode="w+", dtype=np.float32, shape=(len(ecg_ids), 12, 5000))
    for c in range(n_ch):
        a = np.load(CHD / f"chunk_{c:03d}.npy")
        Xm[c * CH:c * CH + len(a)] = a
    Xm.flush(); del Xm
    shutil.copy(Xloc, OUT / "X_raw_12lead.npy")
    shutil.rmtree(CHD)
    XP = Xloc
print("using X from:", XP)

### 2d. Load labels

In [ ]:
y = np.load(YP)
assert len(y) == len(pat)
y_val, y_test = y[val_idx], y[test_idx]
np.save(OUT / "val_idx.npy", val_idx); np.save(OUT / "test_idx.npy", test_idx)
np.save(OUT / "y_val.npy", y_val); np.save(OUT / "y_test.npy", y_test)
d = int(np.abs(np.bincount(y_test, minlength=K) - TEST_SUPPORT).sum())
print("test class counts:", np.bincount(y_test, minlength=K), "| paper:", TEST_SUPPORT, "| L1 diff:", d)
assert d <= 10, "labels do not match the paper"

## 3. Model definitions (architectures unchanged, needed to load the saved weights)

In [ ]:
class MambaBlock(nn.Module):
    def __init__(self, d_model=128, d_state=16, d_conv=4, expand=2, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_inner = int(d_model * expand)
        self.d_state = d_state
        self.in_proj  = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d   = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                  padding=d_conv - 1, groups=self.d_inner, bias=True)
        self.x_proj   = nn.Linear(self.d_inner, d_state * 2 + self.d_inner, bias=False)
        self.dt_proj  = nn.Linear(self.d_inner, self.d_inner, bias=True)
        self.A_log    = nn.Parameter(
            torch.log(torch.arange(1, d_state + 1, dtype=torch.float32)
                      .unsqueeze(0).expand(self.d_inner, -1)))
        self.D        = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.drop     = nn.Dropout(dropout)
        self.norm     = nn.LayerNorm(d_model)

    def ssm_scan(self, x, delta, B, C):
        A     = -torch.exp(self.A_log.float())
        delta = torch.nn.functional.softplus(delta)
        dA    = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))
        dB    = delta.unsqueeze(-1) * B.unsqueeze(2)
        dBx   = dB * x.unsqueeze(-1)
        log_dA_cum = torch.cumsum(torch.log(dA.clamp(min=1e-8)), dim=1)
        dA_cum     = torch.exp(log_dA_cum)
        h = dA_cum * torch.cumsum(dBx / dA_cum.clamp(min=1e-8), dim=1)
        y = (h * C.unsqueeze(2)).sum(dim=-1)
        return y + x * self.D.unsqueeze(0).unsqueeze(0)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        _, L, _ = x.shape
        xz = self.in_proj(x)
        x_b, z = xz.chunk(2, dim=-1)
        x_b = self.conv1d(x_b.permute(0, 2, 1))[..., :L].permute(0, 2, 1)
        x_b = torch.nn.functional.silu(x_b)
        ssm_in = self.x_proj(x_b)
        B_ssm  = ssm_in[..., :self.d_state]
        C_ssm  = ssm_in[..., self.d_state:2 * self.d_state]
        delta  = self.dt_proj(ssm_in[..., 2 * self.d_state:])
        y = self.ssm_scan(x_b, delta, B_ssm, C_ssm)
        y = y * torch.nn.functional.silu(z)
        return self.drop(self.out_proj(y)) + residual


class MambaECG(nn.Module):
    def __init__(self, n_leads=12, n_classes=5, d_model=128, n_layers=4, d_state=16, dropout=0.2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(n_leads, d_model // 2, kernel_size=15, stride=4, padding=7, bias=False),
            nn.BatchNorm1d(d_model // 2), nn.GELU(),
            nn.Conv1d(d_model // 2, d_model, kernel_size=7, stride=4, padding=3, bias=False),
            nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.blocks = nn.ModuleList([
            MambaBlock(d_model=d_model, d_state=d_state, d_conv=4, expand=2, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.norm_out = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(d_model * 2, 256),
            nn.GELU(), nn.Dropout(dropout / 2), nn.Linear(256, n_classes),
        )

    def forward(self, x):
        x = self.stem(x).permute(0, 2, 1)
        for block in self.blocks:
            x = block(x)
        x = self.norm_out(x)
        x = torch.cat([x.mean(dim=1), x.max(dim=1)[0]], dim=1)
        return self.head(x)


class ResidualBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7, stride=1, dropout=0.2):
        super().__init__()
        pad = kernel // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, stride=1, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.drop  = nn.Dropout(dropout)
        self.relu  = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch),
            )

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        if self.downsample:
            identity = self.downsample(x)
        return self.relu(out + identity)


class MultiLead1DCNN(nn.Module):
    def __init__(self, n_leads=12, n_classes=5):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(n_leads, 64, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = nn.Sequential(ResidualBlock1D(64, 64), ResidualBlock1D(64, 64))
        self.layer2 = nn.Sequential(ResidualBlock1D(64, 128, stride=2), ResidualBlock1D(128, 128))
        self.layer3 = nn.Sequential(ResidualBlock1D(128, 256, stride=2), ResidualBlock1D(256, 256))
        self.layer4 = nn.Sequential(ResidualBlock1D(256, 512, stride=2), ResidualBlock1D(512, 512))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.2), nn.Linear(256, n_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        return self.head(self.pool(x).squeeze(-1))


class MultiLeadBiLSTM(nn.Module):
    def __init__(self, n_leads=12, n_classes=5, hidden=192, num_layers=2, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(n_leads, 64, kernel_size=15, stride=4, padding=7, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.Conv1d(64, 96, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(96), nn.ReLU(inplace=True),
            nn.Conv1d(96, 96, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm1d(96), nn.ReLU(inplace=True),
        )
        self.lstm = nn.LSTM(input_size=96, hidden_size=hidden, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden * 2 * 2, 128),
            nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.stem(x).permute(0, 2, 1)
        out, _ = self.lstm(x)
        pooled = torch.cat([out.mean(dim=1), out.max(dim=1)[0]], dim=1)
        return self.head(pooled)


class CRNN(nn.Module):
    def __init__(self, n_leads=12, n_classes=5, conv_ch=128, lstm_hidden=256,
                 n_lstm_layers=2, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_leads, 64, kernel_size=11, stride=2, padding=5, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.Conv1d(64, conv_ch, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(conv_ch), nn.ReLU(inplace=True),
            nn.Conv1d(conv_ch, conv_ch, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm1d(conv_ch), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(input_size=conv_ch, hidden_size=lstm_hidden, num_layers=n_lstm_layers,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.attn = nn.Linear(lstm_hidden * 2, 1)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(lstm_hidden * 2, 128),
            nn.ReLU(inplace=True), nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.conv(x).permute(0, 2, 1)
        out, _ = self.lstm(x)
        attn_w = torch.softmax(self.attn(out), dim=1)
        x = (out * attn_w).sum(dim=1)
        return self.head(x)

## 4. Inference from saved best weights (M1-M5)
Saves per-record softmax probabilities for val and test. Prints test macro F1 next to the paper's value as a sanity check.

In [ ]:
SPECS = {
    "M1": (MambaECG,        "M1_MambaSSM",       0.6238),
    "M2": (MultiLead1DCNN,  "M2_MultiLead1DCNN", 0.6452),
    "M3": (MultiLeadBiLSTM, "M3_BiLSTM",         0.6442),
    "M4": (CRNN,            "M4_CRNN",           0.6376),
}

@torch.no_grad()
def predict(model, X, bs=32):
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(np.ascontiguousarray(X[i:i + bs])).float().to(device)
        out.append(torch.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

need = [f"probs_{s}_{k}.npy" for k in ("M1", "M2", "M3", "M4", "M5") for s in ("val", "test")]
if all((OUT / f).exists() for f in need):
    log.info("M1-M5 probabilities already saved, skipping.")
else:
    X = np.load(XP)
    X_val, X_test = X[val_idx], X[test_idx]
    del X; gc.collect()
    pv, pt = {}, {}
    for key, (cls, ckpt_name, paper_f1) in SPECS.items():
        state = torch.load(RES / f"{ckpt_name}_best.pt", map_location="cpu")
        m = cls(n_leads=12, n_classes=K)
        m.load_state_dict(state)
        m.to(device)
        pv[key], pt[key] = predict(m, X_val), predict(m, X_test)
        f1 = f1_score(y_test, pt[key].argmax(1), average="macro")
        log.info(f"{key}: test macro F1 {f1:.4f} (paper {paper_f1:.4f}) "
                 f"{'OK' if abs(f1 - paper_f1) < 0.002 else 'MISMATCH - CHECK'}")
        del m, state
        torch.cuda.empty_cache()
    pv["M5"] = np.mean([pv[k] for k in ("M2", "M3", "M4")], axis=0)
    pt["M5"] = np.mean([pt[k] for k in ("M2", "M3", "M4")], axis=0)
    f1 = f1_score(y_test, pt["M5"].argmax(1), average="macro")
    log.info(f"M5: test macro F1 {f1:.4f} (paper 0.6651) {'OK' if abs(f1 - 0.6651) < 0.002 else 'MISMATCH - CHECK'}")
    for k in pt:
        np.save(OUT / f"probs_val_{k}.npy", pv[k])
        np.save(OUT / f"probs_test_{k}.npy", pt[k])
    del X_val, X_test; gc.collect()

## 5. M6 fixed
Changes vs the paper: MiniRocket is fit on a subset of **train only** (no test records), SMOTE and the scaler use **train only**, and the number of LightGBM trees is chosen by early stopping on the untouched **validation** set. Other LightGBM settings are fixed a priori (unchanged). Features are cached under a new filename so the old leaky cache is not reused.

The MiniRocket transform takes roughly 20-45 min on CPU.

In [ ]:
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler

ZF = OUT / "Z_minirocket_trainfit.npy"
META = OUT / "m6_meta.json"
meta = json.load(open(META)) if META.exists() else {}

if not ZF.exists():
    X = np.load(XP)
    rng = np.random.default_rng(SEED)
    fit_idx = np.sort(rng.choice(train_idx, 5000, replace=False))   # train records only
    try:
        mr = MiniRocketMultivariate(num_kernels=10000, random_state=SEED, n_jobs=-1)
    except TypeError:
        mr = MiniRocketMultivariate(num_kernels=10000, random_state=SEED)
    t0 = time.time()
    mr.fit(X[fit_idx])
    meta["minirocket_fit_s"] = time.time() - t0
    t0 = time.time()
    chunks = []
    for i in range(0, len(X), 1000):
        chunks.append(np.asarray(mr.transform(X[i:i + 1000]), dtype=np.float32))
        if (i // 1000) % 5 == 0:
            log.info(f"transform {min(i + 1000, len(X))}/{len(X)}")
    Z = np.concatenate(chunks)
    meta["minirocket_transform_s_all_records"] = time.time() - t0
    meta["minirocket_transform_s_per_record"] = meta["minirocket_transform_s_all_records"] / len(X)
    np.save(ZF, Z)
    json.dump(meta, open(META, "w"), indent=2)
    del X, chunks, mr; gc.collect()
else:
    Z = np.load(ZF)
Z = np.nan_to_num(Z, nan=0.0, posinf=0.0, neginf=0.0)
log.info(f"Z {Z.shape}")

def fit_lgbm(Ztr, ytr, Zv, yv, smote):
    if smote:
        Ztr, ytr = SMOTE(random_state=SEED, k_neighbors=3).fit_resample(Ztr, ytr)
    sc = StandardScaler().fit(Ztr)
    clf = lgb.LGBMClassifier(n_estimators=400, num_leaves=31, learning_rate=0.05,
                             colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
                             min_child_samples=20, class_weight="balanced",
                             random_state=SEED, n_jobs=-1, verbose=-1)
    t0 = time.time()
    clf.fit(sc.transform(Ztr), ytr, eval_set=[(sc.transform(Zv), yv)],
            callbacks=[lgb.early_stopping(50, verbose=False)])
    return clf, sc, time.time() - t0, len(ytr)

runs = [("M6", True)]
if RUN_NO_SMOTE_ABLATION:
    runs.append(("M6_noSMOTE", False))

for name, use_smote in runs:
    if (OUT / f"probs_test_{name}.npy").exists():
        log.info(f"{name} already done, skipping.")
        continue
    log.info(f"Training {name} (SMOTE={use_smote}) ...")
    clf, sc, fit_s, n_train = fit_lgbm(Z[train_idx], y[train_idx], Z[val_idx], y_val, use_smote)
    pv = clf.predict_proba(sc.transform(Z[val_idx])).astype(np.float32)
    pt = clf.predict_proba(sc.transform(Z[test_idx])).astype(np.float32)
    np.save(OUT / f"probs_val_{name}.npy", pv)
    np.save(OUT / f"probs_test_{name}.npy", pt)
    meta[name] = dict(lgbm_fit_s=fit_s, n_train_after_resample=int(n_train),
                      best_iteration=int(clf.best_iteration_ or 400))
    json.dump(meta, open(META, "w"), indent=2)
    f1 = f1_score(y_test, pt.argmax(1), average="macro")
    log.info(f"{name}: test macro F1 {f1:.4f} | acc {accuracy_score(y_test, pt.argmax(1)):.4f} | "
             f"trees {meta[name]['best_iteration']} | fit {fit_s / 60:.1f} min")
    del clf, sc; gc.collect()
print(json.dumps(meta, indent=2))

## 6. Statistics
- Bootstrap resamples **patients** (not records) with replacement, and every model is evaluated on the **same** resample, so all comparisons are paired.
- Equivalence: a pair counts as equivalent only if the 90% CI of the macro F1 difference lies inside +/- DELTA (TOST logic).
- McNemar (exact) on per-record correctness, Holm-adjusted across pairs.

In [ ]:
from scipy.stats import binomtest

names = ["M1", "M2", "M3", "M4", "M5", "M6", "M6_noSMOTE"]
models = [m for m in names if (OUT / f"probs_test_{m}.npy").exists()]
P = {m: np.load(OUT / f"probs_test_{m}.npy") for m in models}
yt = y_test
pred = {m: P[m].argmax(1) for m in models}
Y1 = np.eye(K)[yt]
print("models:", models)

def macro_f1_from_cm(cm):
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0.0).mean()

def ece(p, yv, n_bins=15):
    conf, pr = p.max(1), p.argmax(1)
    ok = (pr == yv)
    edges = np.linspace(0, 1, n_bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return e

def holm(pv):
    pv = np.asarray(pv); o = np.argsort(pv); m = len(pv)
    adj = np.empty(m); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (m - r) * pv[i]); adj[i] = min(1.0, run)
    return adj

# ---- patient-cluster paired bootstrap of macro F1
rng = np.random.default_rng(SEED)
_, pat_inv = np.unique(pat[test_idx], return_inverse=True)
n_pat = pat_inv.max() + 1
codes = {m: yt * K + pred[m] for m in models}
F = np.zeros((B_BOOT, len(models)))
for b in range(B_BOOT):
    counts = np.bincount(rng.integers(0, n_pat, n_pat), minlength=n_pat)
    w = counts[pat_inv].astype(float)
    for j, m in enumerate(models):
        cm = np.bincount(codes[m], weights=w, minlength=K * K).reshape(K, K)
        F[b, j] = macro_f1_from_cm(cm)

# ---- summary table
rows = []
for j, m in enumerate(models):
    p = P[m]
    lo, hi = np.percentile(F[:, j], [2.5, 97.5])
    rows.append(dict(
        model=m,
        acc=accuracy_score(yt, pred[m]),
        macro_f1=f1_score(yt, pred[m], average="macro"), f1_lo=lo, f1_hi=hi,
        macro_auroc=np.mean([roc_auc_score(yt == k, p[:, k]) for k in range(K)]),
        macro_auprc=np.mean([average_precision_score(yt == k, p[:, k]) for k in range(K)]),
        ece=ece(p, yt),
        brier=((p - Y1) ** 2).sum(1).mean(),
        nll=-np.log(np.clip(p[np.arange(len(yt)), yt], 1e-12, 1)).mean()))
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "summary.csv", index=False)
print("SUMMARY (95% CI on macro F1 from patient-cluster bootstrap)")
display(summary.round(4))

# ---- per-class table
rows = []
for m in models:
    p = P[m]
    for k, c in enumerate(CLASSES):
        t, q = (yt == k), (pred[m] == k)
        tp = (t & q).sum(); fn = (t & ~q).sum(); fp = (~t & q).sum(); tn = (~t & ~q).sum()
        rows.append(dict(model=m, cls=c, n=int(t.sum()),
                         auroc=roc_auc_score(t, p[:, k]), auprc=average_precision_score(t, p[:, k]),
                         sens=tp / max(tp + fn, 1), spec=tn / max(tn + fp, 1),
                         ppv=tp / max(tp + fp, 1), f1=2 * tp / max(2 * tp + fp + fn, 1)))
perclass = pd.DataFrame(rows)
perclass.to_csv(OUT / "per_class.csv", index=False)
print("PER-CLASS")
display(perclass.round(3))

# ---- pairwise paired comparisons
rows = []
for i in range(len(models)):
    for j in range(i + 1, len(models)):
        a, b_ = models[i], models[j]
        d = F[:, i] - F[:, j]
        point = f1_score(yt, pred[a], average="macro") - f1_score(yt, pred[b_], average="macro")
        ca, cb = pred[a] == yt, pred[b_] == yt
        n10, n01 = int((ca & ~cb).sum()), int((~ca & cb).sum())
        pm = binomtest(min(n10, n01), n10 + n01, 0.5).pvalue if (n10 + n01) > 0 else 1.0
        lo95, hi95 = np.percentile(d, [2.5, 97.5])
        lo90, hi90 = np.percentile(d, [5, 95])
        rows.append(dict(A=a, B=b_, diff_F1=point, ci95_lo=lo95, ci95_hi=hi95,
                         P_A_gt_B=(d > 0).mean(), mcnemar_A_only=n10, mcnemar_B_only=n01, mcnemar_p=pm,
                         equivalent_within_delta=bool(lo90 > -DELTA and hi90 < DELTA)))
pairs = pd.DataFrame(rows)
pairs["mcnemar_p_holm"] = holm(pairs["mcnemar_p"].values)
pairs.to_csv(OUT / "pairwise.csv", index=False)
print(f"PAIRWISE (equivalence margin +/-{DELTA} macro F1, 90% CI inside margin)")
display(pairs.round(4))

# ---- reliability plot
fig, ax = plt.subplots(figsize=(5, 5))
edges = np.linspace(0, 1, 11)
for m in models:
    conf, ok = P[m].max(1), (pred[m] == yt)
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (conf > lo) & (conf <= hi)
        if s.sum() >= 20:
            xs.append(conf[s].mean()); ys.append(ok[s].mean())
    ax.plot(xs, ys, marker="o", ms=3, label=m)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("Mean predicted confidence"); ax.set_ylabel("Accuracy"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "reliability.png", dpi=200)
print("Saved summary.csv, per_class.csv, pairwise.csv, reliability.png to", OUT)

In [ ]:
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt

FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
S  = pd.read_csv(OUT / "summary.csv").set_index("model")
PC = pd.read_csv(OUT / "per_class.csv")
CL = ["CD", "HYP", "MI", "NORM", "STTC"]

COL = {"M1": "#2a78d6", "M2": "#eb6834", "M3": "#1baf7a", "M4": "#eda100",
       "M5": "#e87ba4", "M6_noSMOTE": "#008300", "M6": "#6250d6"}
MARK = {"M1": "o", "M2": "s", "M3": "^", "M4": "D", "M5": "v", "M6_noSMOTE": "P", "M6": "X"}
LAB = {"M1": "M1", "M2": "M2", "M3": "M3", "M4": "M4", "M5": "M5", "M6_noSMOTE": "M6", "M6": "M6 (SMOTE)"}
ORDER = [m for m in ["M1", "M2", "M3", "M4", "M5", "M6_noSMOTE", "M6"] if m in S.index]
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e3e2de"

plt.rcParams.update({"font.family": "serif", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
                     "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7,
                     "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "savefig.dpi": 300,
                     "axes.spines.top": False, "axes.spines.right": False})

def clean(ax, grid_axis="y"):
    ax.grid(axis=grid_axis, color=GRID, lw=0.6); ax.set_axisbelow(True)

# Fig 1: macro F1 with 95% CI
fig, ax = plt.subplots(figsize=(3.5, 2.4))
x = np.arange(len(ORDER))
f1 = np.array([S.at[m, "macro_f1"] for m in ORDER])
lo = f1 - np.array([S.at[m, "f1_lo"] for m in ORDER]); hi = np.array([S.at[m, "f1_hi"] for m in ORDER]) - f1
ax.bar(x, f1, width=0.62, color=[COL[m] for m in ORDER], edgecolor="white", linewidth=1.0)
ax.errorbar(x, f1, yerr=[lo, hi], fmt="none", ecolor=INK, elinewidth=0.9, capsize=2.2, capthick=0.9)
for xi, v, h in zip(x, f1, hi):
    ax.text(xi, v + h + 0.004, f"{v:.3f}", ha="center", va="bottom", fontsize=6.8, color=INK)
ax.set_xticks(x); ax.set_xticklabels([LAB[m].replace(" (SMOTE)", "\n(SMOTE)") for m in ORDER])
ax.set_ylim(0.55, 0.72); ax.set_ylabel("Macro F1 (95% CI)")
clean(ax); fig.tight_layout(); fig.savefig(FIG / "fig_macro_f1.png"); plt.close(fig)

# Fig 2, 3: confusion matrices (M5 and M6 without SMOTE)
def cm_fig(model, title, fname):
    p = np.load(OUT / f"probs_test_{model}.npy").argmax(1)
    cm = np.zeros((5, 5), int)
    np.add.at(cm, (y_test, p), 1)
    rn = cm / cm.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(3.4, 3.1))
    im = ax.imshow(rn, cmap="Blues", vmin=0, vmax=1)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{cm[i, j]}\n{rn[i, j]*100:.0f}%", ha="center", va="center", fontsize=6.5,
                    color="white" if rn[i, j] > 0.55 else INK)
    ax.set_xticks(range(5)); ax.set_xticklabels(CL); ax.set_yticks(range(5)); ax.set_yticklabels(CL)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title, fontsize=8)
    for s in ax.spines.values(): s.set_visible(False)
    ax.tick_params(length=0)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).ax.tick_params(labelsize=6.5)
    fig.tight_layout(); fig.savefig(FIG / fname); plt.close(fig)
cm_fig("M5", "M5 ensemble", "fig_cm_M5.png")
cm_fig("M6_noSMOTE", "M6 MiniRocket + LightGBM", "fig_cm_M6.png")

# Fig 4: per-class F1
show = [m for m in ["M1", "M2", "M3", "M4", "M5", "M6_noSMOTE"] if m in S.index]
fig, ax = plt.subplots(figsize=(7.1, 2.5))
w = 0.8 / len(show)
for k, m in enumerate(show):
    d = PC[PC.model == m].set_index("cls")
    vals = [d.at[c, "f1"] for c in CL]
    xs = np.arange(5) + (k - (len(show) - 1) / 2) * w
    ax.bar(xs, vals, width=w * 0.88, color=COL[m], edgecolor="white", linewidth=1.0, label=LAB[m])
    for xi, v in zip(xs, vals):
        ax.text(xi, v + 0.008, f"{v:.2f}", ha="center", va="bottom", fontsize=5.8, rotation=90, color=INK)
ax.set_xticks(range(5)); ax.set_xticklabels(CL); ax.set_ylim(0, 1.0); ax.set_ylabel("F1 (argmax)")
ax.legend(ncol=6, loc="upper center", bbox_to_anchor=(0.5, 1.16), frameon=False, handlelength=1.2)
clean(ax); fig.tight_layout(); fig.savefig(FIG / "fig_perclass_f1.png"); plt.close(fig)

# Fig 5: reliability diagram
fig, ax = plt.subplots(figsize=(3.4, 3.2))
edges = np.linspace(0, 1, 11)
for m in ORDER:
    p = np.load(OUT / f"probs_test_{m}.npy"); conf = p.max(1); ok = p.argmax(1) == y_test
    xs, ys = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        s = (conf > a) & (conf <= b)
        if s.sum() >= 20:
            xs.append(conf[s].mean()); ys.append(ok[s].mean())
    ax.plot(xs, ys, color=COL[m], marker=MARK[m], ms=3.2, lw=1.1, label=LAB[m])
ax.plot([0, 1], [0, 1], color=MUTED, lw=0.8, ls="--")
ax.set_xlim(0.3, 1.0); ax.set_ylim(0.2, 1.02)
ax.set_xlabel("Mean predicted confidence"); ax.set_ylabel("Accuracy")
ax.legend(frameon=False, loc="lower right", ncol=2, handlelength=1.6)
clean(ax, "both"); fig.tight_layout(); fig.savefig(FIG / "fig_reliability.png"); plt.close(fig)
print("Wrote:", sorted(p.name for p in FIG.glob("*.png")), "->", FIG)

In [ ]:
for m in ["M6", "M6_noSMOTE"]:
    pv = np.load(OUT / f"probs_val_{m}.npy")
    print(m, f1_score(y_val, pv.argmax(1), average="macro"))

In [ ]:
import pandas as pd, numpy as np, json
OUT_TEX = OUT / "tables"; OUT_TEX.mkdir(exist_ok=True)
S  = pd.read_csv(OUT / "summary.csv").set_index("model")
PC = pd.read_csv(OUT / "per_class.csv")
PW = pd.read_csv(OUT / "pairwise.csv")
MT = json.load(open(OUT / "m6_meta.json"))

NAME = {"M1": "M1 Mamba SSM", "M2": "M2 1D ResNet", "M3": "M3 BiLSTM", "M4": "M4 CRNN+Attn",
        "M5": "M5 Ensemble", "M6_noSMOTE": "M6 MiniRocket+LGBM", "M6": "M6$_\\mathrm{SMOTE}$"}
ORDER = ["M1", "M2", "M3", "M4", "M5", "M6_noSMOTE", "M6"]
ORDER = [m for m in ORDER if m in S.index]
f = lambda x, d=3: f"{x:.{d}f}"

# ---- Table I: aggregate
mp = PC.groupby("model")["ppv"].mean(); mr = PC.groupby("model")["sens"].mean()
rows = []
for m in ORDER:
    c = [f(S.at[m, "acc"]),
         f"{f(S.at[m, 'macro_f1'])} [{f(S.at[m, 'f1_lo'])}, {f(S.at[m, 'f1_hi'])}]",
         f(mp[m]), f(mr[m]), f(S.at[m, "macro_auroc"]), f(S.at[m, "macro_auprc"]), f(S.at[m, "ece"])]
    rows.append(NAME[m] + " & " + " & ".join(c) + " \\\\")
t1 = ("\\begin{table*}[t]\n\\caption{Aggregate test-set performance (3,182 records). Macro F1 95\\% CI from a patient-level bootstrap "
      "(10,000 resamples). Precision/recall/F1 at argmax; AUROC/AUPRC macro one-vs-rest; ECE with 15 bins.}\n"
      "\\label{tab:agg}\n\\centering\n\\begin{tabular}{lccccccc}\n\\hline\n"
      "Model & Acc. & Macro F1 [95\\% CI] & Macro Prec. & Macro Rec. & Macro AUROC & Macro AUPRC & ECE \\\\\n\\hline\n"
      + "\n".join(rows) + "\n\\hline\n\\end{tabular}\n\\end{table*}\n")
(OUT_TEX / "table_aggregate.tex").write_text(t1)

# ---- Table: per-class AUROC and F1
CL = ["CD", "HYP", "MI", "NORM", "STTC"]
rows = []
for m in ORDER:
    d = PC[PC.model == m].set_index("cls")
    rows.append(NAME[m] + " & " + " & ".join(f(d.at[c, "auroc"], 2) for c in CL) + " & "
                + " & ".join(f(d.at[c, "f1"], 2) for c in CL) + " \\\\")
t2 = ("\\begin{table*}[t]\n\\caption{Per-class one-vs-rest AUROC and argmax F1 on the test set.}\n\\label{tab:perclass}\n\\centering\n"
      "\\begin{tabular}{l" + "c" * 10 + "}\n\\hline\n & \\multicolumn{5}{c}{AUROC} & \\multicolumn{5}{c}{F1 (argmax)} \\\\\n"
      "Model & " + " & ".join(CL) + " & " + " & ".join(CL) + " \\\\\n\\hline\n" + "\n".join(rows) + "\n\\hline\n\\end{tabular}\n\\end{table*}\n")
(OUT_TEX / "table_perclass.tex").write_text(t2)

# ---- Table: HYP detail
rows = []
for m in ORDER:
    d = PC[(PC.model == m) & (PC.cls == "HYP")].iloc[0]
    rows.append(NAME[m] + " & " + " & ".join(f(d[k]) for k in ["auroc", "auprc", "sens", "spec", "ppv", "f1"]) + " \\\\")
t3 = ("\\begin{table}[t]\n\\caption{Hypertrophy (HYP) detail on the test set (208 records). Sensitivity, specificity, PPV and F1 at argmax.}\n"
      "\\label{tab:hyp}\n\\centering\n\\begin{tabular}{lcccccc}\n\\hline\nModel & AUROC & AUPRC & Sens. & Spec. & PPV & F1 \\\\\n\\hline\n"
      + "\n".join(rows) + "\n\\hline\n\\end{tabular}\n\\end{table}\n")
(OUT_TEX / "table_hyp.tex").write_text(t3)

# ---- Table: key paired comparisons (macro F1 difference A-B)
def get(a, b):
    r = PW[(PW.A == a) & (PW.B == b)]
    if len(r):
        r = r.iloc[0]; return r.diff_F1, r.ci95_lo, r.ci95_hi, r.mcnemar_p_holm, r.equivalent_within_delta
    r = PW[(PW.A == b) & (PW.B == a)].iloc[0]
    return -r.diff_F1, -r.ci95_hi, -r.ci95_lo, r.mcnemar_p_holm, r.equivalent_within_delta
pairs = [("M5", "M1"), ("M5", "M2"), ("M5", "M3"), ("M5", "M4"), ("M5", "M6_noSMOTE"), ("M5", "M6"),
         ("M2", "M1"), ("M3", "M1"), ("M4", "M1"), ("M2", "M3"), ("M6_noSMOTE", "M6")]
pairs = [p for p in pairs if p[0] in S.index and p[1] in S.index]
short = lambda m: {"M6_noSMOTE": "M6", "M6": "M6$_\\mathrm{SMOTE}$"}.get(m, m)
rows = []
for a, b in pairs:
    d, lo, hi, ph, eq = get(a, b)
    ph = "$<$0.001" if ph < 0.001 else f(ph)
    rows.append(f"{short(a)} $-$ {short(b)} & {d:+.3f} & [{lo:+.3f}, {hi:+.3f}] & {ph} & {'yes' if eq else 'no'} \\\\")
t4 = ("\\begin{table}[t]\n\\caption{Paired comparisons of macro F1 (A $-$ B). 95\\% CI from the patient-level paired bootstrap; "
      "McNemar $p$ (exact, Holm-adjusted over all 21 pairs) tests per-record correctness; equivalence = 90\\% CI inside $\\pm$0.02.}\n"
      "\\label{tab:pairs}\n\\centering\n\\begin{tabular}{lcccc}\n\\hline\nA $-$ B & $\\Delta$F1 & 95\\% CI & McNemar $p_\\mathrm{Holm}$ & Equiv. \\\\\n\\hline\n"
      + "\n".join(rows) + "\n\\hline\n\\end{tabular}\n\\end{table}\n")
(OUT_TEX / "table_pairs.tex").write_text(t4)

# ---- compute cost for M6 + patient count check
tr, fit0, fit1 = MT["minirocket_transform_s_all_records"], MT["M6_noSMOTE"]["lgbm_fit_s"], MT["M6"]["lgbm_fit_s"]
print(f"MiniRocket transform: {tr/60:.1f} min for all records = {MT['minirocket_transform_s_per_record']:.3f} s/record (CPU)")
print(f"M6 (no SMOTE): fit {fit0/60:.1f} min | pipeline total {(MT['minirocket_fit_s']+tr+fit0)/60:.1f} min")
print(f"M6 (SMOTE):    fit {fit1/60:.1f} min | pipeline total {(MT['minirocket_fit_s']+tr+fit1)/60:.1f} min")
print("patients in cohort:", len(np.unique(pat)), "| test records:", len(test_idx))
print("Wrote:", sorted(p.name for p in OUT_TEX.glob("*.tex")), "->", OUT_TEX)
for p in sorted(OUT_TEX.glob("*.tex")):
    print("\n%%", p.name); print(p.read_text())

In [ ]:
for m in ["M1","M2","M3","M4"]:
    pv = np.load(OUT / f"probs_val_{m}.npy"); print(m, round(f1_score(y_val, pv.argmax(1), average="macro"), 4))

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
V = {m: np.load(OUT / f"probs_val_{m}.npy") for m in ["M1", "M2", "M3", "M4"]}
P = {m: np.load(OUT / f"probs_test_{m}.npy") for m in ["M1", "M2", "M3", "M4"]}
vf = {m: f1_score(y_val, V[m].argmax(1), average="macro") for m in V}
top3 = sorted(sorted(vf, key=vf.get, reverse=True)[:3])
combos = {"M5 as in paper (M2+M3+M4)": ["M2", "M3", "M4"],
          "top 3 by VALIDATION (" + "+".join(top3) + ")": top3,
          "all four deep (M1+M2+M3+M4)": ["M1", "M2", "M3", "M4"]}

def mf1(cm):
    tp = np.diag(cm); den = 2 * tp + (cm.sum(0) - tp) + (cm.sum(1) - tp)
    return np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0.0).mean()

_, inv = np.unique(pat[test_idx], return_inverse=True); n = inv.max() + 1
rng = np.random.default_rng(42)
W = [np.bincount(rng.integers(0, n, n), minlength=n)[inv].astype(float) for _ in range(2000)]
pred, codes = {}, {}
for name, ms in combos.items():
    pv = np.mean([V[m] for m in ms], 0); pt = np.mean([P[m] for m in ms], 0)
    pred[name] = pt.argmax(1); codes[name] = y_test * K + pred[name]
    print(f"{name:42s} val F1 {f1_score(y_val, pv.argmax(1), average='macro'):.4f} | "
          f"test F1 {f1_score(y_test, pred[name], average='macro'):.4f} | test acc {accuracy_score(y_test, pred[name]):.4f}")
ref = list(combos)[0]
print("\nValidation macro F1 per model:", {m: round(v, 4) for m, v in vf.items()})
print(f"\nDifference vs the paper's M5 (patient-level paired bootstrap, 2000 resamples):")
for name in list(combos)[1:]:
    d = np.array([mf1(np.bincount(codes[name], weights=w, minlength=K * K).reshape(K, K)) -
                  mf1(np.bincount(codes[ref], weights=w, minlength=K * K).reshape(K, K)) for w in W])
    print(f"  {name:42s} {np.mean(d):+.4f}  95% CI [{np.percentile(d, 2.5):+.4f}, {np.percentile(d, 97.5):+.4f}]")